In [88]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import SAGEConv, Node2Vec
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dataset = Planetoid(root="data/Planetoid", name="CiteSeer")

data = dataset[0]

C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\torch_geometric\data\dataset.py:238: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  if osp.exists(f) and torch.lo

In [89]:
transform = RandomLinkSplit(
    num_val=0.05,
    num_test=0.1,
    is_undirected=True,
    add_negative_train_samples=False
)

train_data, val_data, test_data = transform(data)

train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)

In [90]:
def train_deepwalk():

    dw_model.train()
    total_loss = 0

    for pos_rw, neg_rw in loader:

        dw_optimizer.zero_grad()

        loss = dw_model.loss(pos_rw.to(device), neg_rw.to(device))

        loss.backward()
        dw_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)



@torch.no_grad()
def evaluate_deepwalk(data):

    dw_model.eval()

    z = dw_model()
    z = F.normalize(z, p=2, dim=1)

    out = (z[data.edge_label_index[0]] * z[data.edge_label_index[1]]).sum(dim=-1).sigmoid()

    y_true = data.edge_label.cpu().numpy()
    y_pred = out.cpu().numpy()

    auc = roc_auc_score(y_true, y_pred)
    ap = average_precision_score(y_true, y_pred)

    return auc, ap

In [91]:
dw_model = Node2Vec(
        edge_index=train_data.edge_index,
        embedding_dim=64,
        walk_length=22,
        context_size=9,
        walks_per_node=12,
        num_negative_samples=1,
        p=1.0,
        q=1.0,                 
        num_nodes=train_data.num_nodes,
        sparse=True
    ).to(device)

loader = dw_model.loader(batch_size=256, shuffle=True)
dw_optimizer = torch.optim.SparseAdam(dw_model.parameters(), lr=0.01)

In [92]:
for epoch in range(1, 101):

    loss = train_deepwalk()

    if epoch % 5 == 0:

        val_auc, val_ap = evaluate_deepwalk(val_data)

        print(f"Epoch {epoch:03d} | Loss {loss:.4f} | Val AUC {val_auc:.4f} | Val AP {val_ap:.4f}")
        dw_auc, dw_ap = evaluate_deepwalk(test_data)

        print("\nDeepWalk Test")
        print("AUC:", dw_auc)
        print("AP :", dw_ap)


dw_auc, dw_ap = evaluate_deepwalk(test_data)

print("\nDeepWalk Test")
print("AUC:", dw_auc)
print("AP :", dw_ap)

Epoch 005 | Loss 2.5167 | Val AUC 0.6265 | Val AP 0.5841

DeepWalk Test
AUC: 0.6234995773457311
AP : 0.5910946877904396
Epoch 010 | Loss 1.5504 | Val AUC 0.6897 | Val AP 0.6826

DeepWalk Test
AUC: 0.6984373867890353
AP : 0.6887299675137935
Epoch 015 | Loss 1.1546 | Val AUC 0.7311 | Val AP 0.7488

DeepWalk Test
AUC: 0.7522569737954354
AP : 0.7760155056901745
Epoch 020 | Loss 0.9882 | Val AUC 0.7560 | Val AP 0.7795

DeepWalk Test
AUC: 0.7861514309865958
AP : 0.8303814275341881
Epoch 025 | Loss 0.9077 | Val AUC 0.7762 | Val AP 0.7994

DeepWalk Test
AUC: 0.8041540876705712
AP : 0.8557292891987429
Epoch 030 | Loss 0.8625 | Val AUC 0.7843 | Val AP 0.8066

DeepWalk Test
AUC: 0.8112208670450429
AP : 0.8640258353939687
Epoch 035 | Loss 0.8341 | Val AUC 0.7864 | Val AP 0.8183

DeepWalk Test
AUC: 0.8138316628426518
AP : 0.8674598587344321
Epoch 040 | Loss 0.8146 | Val AUC 0.7893 | Val AP 0.8244

DeepWalk Test
AUC: 0.8180026566839754
AP : 0.8717878986892853
Epoch 045 | Loss 0.8011 | Val AUC 0.7919

In the above cell after running a few times the training with different configurations and monitoring the outputs I picked the best number of epochs and configuration.

In [93]:
print("\nTraining DeepWalk")

num_runs = 50
test_aucs = []
test_aps = []

for run in range(1, num_runs + 1):

    dw_model = Node2Vec(
    train_data.edge_index,
    embedding_dim=64,
    walk_length=22,
    context_size=9,
    walks_per_node=12,
    num_negative_samples=1,
    p=1,
    q=1,
    sparse=True
).to(device)

    loader = dw_model.loader(batch_size=256, shuffle=True)
    dw_optimizer = torch.optim.SparseAdam(dw_model.parameters(), lr=0.01)
    for epoch in range(1, 61):
        loss = train_deepwalk()
        
    dw_auc, dw_ap = evaluate_deepwalk(test_data)
    test_aucs.append(dw_auc)
    test_aps.append(dw_ap)
    print(f"Run {run:02d} Completed - Test AUC: {dw_auc:.4f}, Test AP: {dw_ap:.4f}")

print(f"\n--- Final Results over {num_runs} runs ---")
print(f"Test AUC: {np.mean(test_aucs):.4f} ± {np.std(test_aucs):.4f}")
print(f"Test AP:  {np.mean(test_aps):.4f} ± {np.std(test_aps):.4f}")


Training DeepWalk
Run 01 Completed - Test AUC: 0.8128, Test AP: 0.8711
Run 02 Completed - Test AUC: 0.8333, Test AP: 0.8815
Run 03 Completed - Test AUC: 0.8364, Test AP: 0.8824
Run 04 Completed - Test AUC: 0.8288, Test AP: 0.8790
Run 05 Completed - Test AUC: 0.8227, Test AP: 0.8759
Run 06 Completed - Test AUC: 0.8398, Test AP: 0.8843
Run 07 Completed - Test AUC: 0.8203, Test AP: 0.8747
Run 08 Completed - Test AUC: 0.8201, Test AP: 0.8750
Run 09 Completed - Test AUC: 0.8280, Test AP: 0.8785
Run 10 Completed - Test AUC: 0.8417, Test AP: 0.8874
Run 11 Completed - Test AUC: 0.8142, Test AP: 0.8726
Run 12 Completed - Test AUC: 0.8384, Test AP: 0.8828
Run 13 Completed - Test AUC: 0.8340, Test AP: 0.8814
Run 14 Completed - Test AUC: 0.8318, Test AP: 0.8789
Run 15 Completed - Test AUC: 0.8330, Test AP: 0.8816
Run 16 Completed - Test AUC: 0.8397, Test AP: 0.8831
Run 17 Completed - Test AUC: 0.8273, Test AP: 0.8782
Run 18 Completed - Test AUC: 0.8148, Test AP: 0.8727
Run 19 Completed - Test AUC